# 01 — Bronze | Credit Risk Ingestion
**Microsoft Fabric · Lakehouse · PySpark**

Objetivo: ler o CSV `credit_risk_dataset.csv` colocado em `Files/landing/` do Lakehouse e persistir uma cópia bruta na camada Bronze em Delta.

**Pré-requisitos**
1. Criar/abrir o Lakehouse `lh_credit_risk` no workspace Fabric.
2. Carregar o ficheiro Kaggle para `Files/landing/credit_risk_dataset.csv`.
3. Anexar este Lakehouse ao notebook e defini-lo como lakehouse padrão.

A camada Bronze preserva os dados recebidos. Não fazemos limpeza nem removemos registos nesta etapa.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

SOURCE_PATH = "Files/landing/credit_risk_dataset.csv"
BRONZE_TABLE = "bronze_credit_risk"

In [ ]:
# Verifica se o ficheiro está acessível no Lakehouse
display(mssparkutils.fs.ls("Files/landing"))

In [ ]:
# Leitura inicial como texto inferido pelo Spark.
# O ficheiro de origem usa cabeçalho e separador vírgula.
df_source = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(SOURCE_PATH)
)

print(f"Linhas: {df_source.count():,}")
print(f"Colunas: {len(df_source.columns)}")
display(df_source.limit(10))
df_source.printSchema()

In [ ]:
# Metadados de ingestão para rastreabilidade
df_bronze = (
    df_source
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit(SOURCE_PATH))
    .withColumn("_source_system", F.lit("Kaggle - Credit Risk Dataset"))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Tabela {BRONZE_TABLE} gravada com sucesso.")

In [ ]:
# Validação básica da carga
bronze = spark.table(BRONZE_TABLE)
print("Total Bronze:", bronze.count())
print("Colunas:", bronze.columns)
display(bronze.limit(10))

## Resultado esperado
- Tabela Delta `bronze_credit_risk` criada no Lakehouse.
- Metadados de ingestão disponíveis.
- Dados originais preservados para auditoria.

**Nota:** `mode("overwrite")` é adequado para a primeira carga/demonstração. Para cargas recorrentes, definir uma estratégia incremental e uma chave de negócio antes de substituir o modo de escrita.